#  AI Image Generator
**Run Cell 1 → Cell 2 → Cell 3 in order.**

> **GPU required:** Runtime → Change runtime type → T4 GPU → Save → Disconnect and delete runtime → reconnect → run all cells


In [ ]:
# @title  Cell 1 — Install
import subprocess, sys
print('Installing...')
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'diffusers>=0.27.0','transformers>=4.38.0','accelerate>=0.27.0',
    'safetensors','Pillow','xformers'])
print(' Done! Run Cell 2.')


In [ ]:
# @title Cell 2 — Load model (VRAM-optimised for T4 15GB)
import torch, os, gc
from diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline, DPMSolverMultistepScheduler
from IPython.display import display, HTML
import ipywidgets as w, io as _io, base64, time, traceback

os.makedirs('/content/generated_images', exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Go to Runtime > Change runtime type > T4 GPU')

gpu_name = torch.cuda.get_device_name(0)
vram     = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name}  ({vram:.1f} GB VRAM)')

DEVICE = 'cuda'
DTYPE  = torch.float16

# Helps prevent fragmentation OOM during VAE decode
import os as _os
_os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

gc.collect(); torch.cuda.empty_cache()

def _sched(cfg):
    return DPMSolverMultistepScheduler.from_config(
        cfg, use_karras_sigmas=True, algorithm_type='dpmsolver++'
    )

# Load base onto GPU
print('Loading SDXL Base (~14 GB first run)...')
base = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=DTYPE, variant='fp16', use_safetensors=True,
).to(DEVICE)
base.scheduler = _sched(base.scheduler.config)
base.enable_attention_slicing()
# VAE tiling: decodes in tiles so the 1GB VAE decode spike never OOMs
base.enable_vae_tiling()
try: base.enable_xformers_memory_efficient_attention()
except: pass

# Refiner: use sequential CPU offload so it only occupies GPU during its pass
# This frees ~6 GB vs loading both to GPU simultaneously
print('Loading SDXL Refiner (CPU offload)...')
refiner = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-refiner-1.0',
    torch_dtype=DTYPE, variant='fp16', use_safetensors=True,
)
refiner.scheduler = _sched(refiner.scheduler.config)
refiner.enable_sequential_cpu_offload()   # streams layers GPU<->CPU on demand
refiner.enable_attention_slicing()
refiner.enable_vae_tiling()
try: refiner.enable_xformers_memory_efficient_attention()
except: pass

gc.collect(); torch.cuda.empty_cache()
used = torch.cuda.memory_allocated() / 1024**3
print(f'Ready!  VRAM used by base: {used:.1f} / {vram:.1f} GB')
print('Run Cell 3 to open the UI.')


In [ ]:
# @title  Cell 3 — Generator UI

from IPython.display import display, clear_output, HTML
import ipywidgets as w, io as _io, base64, time, gc, traceback

STYLES = {
  'Photorealistic': {
    'suffix': ', RAW photo, photorealistic, 8K UHD, DSLR, sharp focus, high detail, natural lighting',
    'neg': 'painting, illustration, anime, cartoon, CGI, render, low quality, blurry, watermark',
    'cfg': 8.5},
  'Cinematic': {
    'suffix': ', cinematic shot, anamorphic lens, film grain, dramatic lighting, color graded, movie still',
    'neg': 'amateur, low quality, blurry, flat lighting, distorted, watermark, text',
    'cfg': 8.0},
  'Digital Art': {
    'suffix': ', digital art, concept art, artstation, highly detailed, vibrant colors, sharp',
    'neg': 'photo, realistic, low quality, blurry, sketch, watermark',
    'cfg': 7.5},
  'Oil Painting': {
    'suffix': ', oil painting, impasto, master painter, rich textures, dramatic brushstrokes, gallery quality',
    'neg': 'photo, digital art, anime, low quality, blurry, watermark',
    'cfg': 8.0},
  'Anime': {
    'suffix': ', anime style, studio ghibli inspired, vibrant colors, detailed background, beautiful lighting',
    'neg': 'photo, realistic, low quality, blurry, ugly, deformed, watermark',
    'cfg': 7.5},
  'Fantasy Art': {
    'suffix': ', epic fantasy illustration, intricate details, magical atmosphere, dramatic lighting',
    'neg': 'photo, low quality, blurry, modern, ugly, deformed, watermark',
    'cfg': 8.0},
  'Watercolor': {
    'suffix': ', watercolor painting, soft washes, loose brushwork, delicate details, artistic',
    'neg': 'photo, digital, harsh lines, low quality, ugly, watermark',
    'cfg': 7.0},
  'No Style': {
    'suffix': '',
    'neg': 'low quality, blurry, distorted, ugly, deformed, watermark, text',
    'cfg': 7.5},
}

EXAMPLES = [
  (' Garden',    'A serene Japanese garden with cherry blossoms, koi pond and stone lanterns'),
  ('Cyberpunk', 'Futuristic cyberpunk city at night, neon lights, rain-slicked streets'),
  ('Mountains', 'Majestic mountain landscape at golden hour, alpine lake, mirror reflections'),
  (' Dragon',    'Majestic dragon perched on a cliff, glowing amber eyes, epic atmosphere'),
  (' Nebula',    'Breathtaking nebula in deep space, swirling clouds of purple and gold'),
  ('Wizard',    'Old wizard in a tower study, floating spell books and glowing potions'),
  ('Forest',    'Enchanted forest path, bioluminescent flowers, fireflies, misty atmosphere'),
  ('Ruins',     'Ancient temple ruins overgrown with jungle vines, golden sunlight shafts'),
]

# ── CSS ──────────────────────────────────────────────────────────────────
display(HTML('''
<style>
.widget-label { color: #a78bfa !important; font-weight: 700 !important; font-size: 11px !important; text-transform: uppercase; letter-spacing: .06em; }
.widget-text input, .widget-textarea textarea, .widget-dropdown select, .widget-select select {
  background: #0a0a1a !important; color: #e2e8f0 !important;
  border: 1.5px solid #2a2a50 !important; border-radius: 8px !important;
  font-size: 13px !important; padding: 6px 10px !important; }
.widget-text input:focus, .widget-textarea textarea:focus {
  border-color: #8b5cf6 !important; outline: none !important; }
.widget-slider .ui-slider { background: #2a2a50 !important; }
.widget-slider .ui-slider-handle { background: #8b5cf6 !important; border-color: #8b5cf6 !important; }
.widget-button button { border-radius: 8px !important; font-size: 12px !important; }
.ex-btn button { background: #1a1a35 !important; color: #a78bfa !important; border: 1.5px solid #2a2a50 !important; }
.ex-btn button:hover { background: #8b5cf6 !important; color: #fff !important; border-color: #8b5cf6 !important; }
.gen-btn button { background: linear-gradient(135deg,#7c3aed,#4f46e5) !important; color: #fff !important;
  border: none !important; font-size: 15px !important; font-weight: 700 !important;
  height: 52px !important; letter-spacing: .04em !important; }
.gen-btn button:disabled { opacity: .5 !important; }
.sty-btn button { background: #13132a !important; color: #64748b !important; border: 1.5px solid #2a2a50 !important; font-size: 11px !important; }
.sty-btn.active button { background: #8b5cf6 !important; color: #fff !important; border-color: #8b5cf6 !important; }
.card { background:#0d0d1a; border:1px solid #2a2a50; border-radius:14px; padding:20px; max-width:900px; }
.sec-hdr { color:#64748b; font-size:10px; font-weight:700; text-transform:uppercase; letter-spacing:.08em; margin:14px 0 6px; }
</style>
'''))

# ── Widgets ───────────────────────────────────────────────────────────────
title_html = w.HTML('<div style="background:linear-gradient(135deg,#7c3aed,#4f46e5);border-radius:12px;padding:18px 22px;margin-bottom:16px;"><h2 style="color:#fff;margin:0;font-size:20px;">🎨 AI Image Generator</h2><p style="color:#c4b5fd;margin:4px 0 0;font-size:12px;">GPU · SDXL Base + Refiner · High Quality</p></div>')

prompt_box = w.Textarea(
    placeholder='Describe your image in detail... (e.g. "A lone astronaut on Mars at sunset, cinematic")',
    layout=w.Layout(width='100%', height='90px'),
)

# Style toggle buttons
style_state = {'current': 'Photorealistic'}
style_btns  = {s: w.Button(description=s, layout=w.Layout(width='auto')) for s in STYLES}
for b in style_btns.values(): b.add_class('sty-btn')
style_btns['Photorealistic'].add_class('active')

def on_style(s):
    def _cb(b):
        old = style_state['current']
        style_btns[old].remove_class('active')
        style_state['current'] = s
        style_btns[s].add_class('active')
        neg_box.value = STYLES[s]['neg']
        guidance_sl.value = STYLES[s]['cfg']
    return _cb
for s, b in style_btns.items(): b.on_click(on_style(s))
style_row1 = w.HBox(list(style_btns.values())[:4], layout=w.Layout(gap='6px', flex_wrap='wrap'))
style_row2 = w.HBox(list(style_btns.values())[4:], layout=w.Layout(gap='6px', flex_wrap='wrap'))

neg_box = w.Textarea(
    value='painting, illustration, anime, cartoon, CGI, render, low quality, blurry, distorted, watermark',
    layout=w.Layout(width='100%', height='60px'),
)

res_drop    = w.Dropdown(options=['512x512','768x768','1024x1024'], value='1024x1024',
                         description='Resolution', style={'description_width':'80px'},
                         layout=w.Layout(width='220px'))
steps_sl    = w.IntSlider(value=35, min=20, max=60, description='Steps',
                          style={'description_width':'50px'}, layout=w.Layout(width='260px'))
guidance_sl = w.FloatSlider(value=8.5, min=4.0, max=13.0, step=0.5, description='Guidance',
                            style={'description_width':'70px'}, layout=w.Layout(width='260px'))
seed_box    = w.IntText(value=-1, description='Seed (-1)', style={'description_width':'80px'},
                        layout=w.Layout(width='200px'))

# Example buttons
ex_btns = []
for name, text in EXAMPLES:
    b = w.Button(description=name, layout=w.Layout(width='auto'))
    b.add_class('ex-btn')
    def _mk(t):
        def _cb(b): prompt_box.value = t
        return _cb
    b.on_click(_mk(text))
    ex_btns.append(b)
ex_row = w.HBox(ex_btns, layout=w.Layout(flex_wrap='wrap', gap='6px'))

gen_btn = w.Button(description=' Generate Image', layout=w.Layout(width='100%', height='52px'))
gen_btn.add_class('gen-btn')

status_out = w.Output()
img_out    = w.Output()
hist_out   = w.Output()
history    = []
MAX_HIST   = 10

def _b64(img):
    buf = _io.BytesIO()
    img.save(buf, 'PNG')
    return base64.b64encode(buf.getvalue()).decode()

def render_history():
    with hist_out:
        clear_output(wait=True)
        if not history: return
        display(w.HTML('<div class="sec-hdr">History</div>'))
        for img, prompt, elapsed, seed, res, style in reversed(history):
            b64   = _b64(img)
            short = prompt[:70]+'...' if len(prompt)>70 else prompt
            display(HTML(
                f'<div style="margin:8px 0;border:1px solid #2a2a50;border-radius:10px;overflow:hidden;max-width:900px;">'
                f'<img src="data:image/png;base64,{b64}" style="width:100%;display:block;">'
                f'<div style="padding:8px 14px;background:#13132a;display:flex;justify-content:space-between;align-items:center;">'
                f'<span style="color:#94a3b8;font-size:12px;">{short}</span>'
                f'<span style="color:#64748b;font-size:11px;">{style} · {res} · {elapsed:.1f}s · seed {seed}</span>'
                f'</div>'
                f'<a download="img_{seed}.png" href="data:image/png;base64,{b64}"'
                f' style="display:block;padding:10px;text-align:center;background:#4f46e5;color:#fff;text-decoration:none;font-size:13px;font-weight:700;">⬇ Download</a>'
                f'</div>'
            ))

def on_generate(b):
    prompt = prompt_box.value.strip()
    if not prompt:
        with status_out:
            clear_output(wait=True)
            display(HTML('<p style="color:#ef4444;margin:0;">Please enter a prompt first!</p>'))
        return

    style  = style_state['current']
    neg    = neg_box.value.strip()
    res    = res_drop.value
    steps  = steps_sl.value
    guid   = guidance_sl.value
    seed   = seed_box.value
    w_val, h_val = map(int, res.split('x'))

    gen_btn.disabled    = True
    gen_btn.description = '  Generating...'

    with status_out:
        clear_output(wait=True)
        vram_free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3
        display(HTML(f'<p style="color:#f59e0b;margin:0;">Generating {res} · {steps} steps · style: {style} · VRAM free: {vram_free:.1f} GB</p>'))

    gc.collect()
    torch.cuda.empty_cache()

    final_prompt = prompt + STYLES[style]['suffix']

    try:
        gen  = torch.Generator(device=DEVICE)
        used = int(torch.randint(0, 2**32, (1,)).item()) if seed == -1 else seed
        gen.manual_seed(used)

        t0 = time.time()
        with torch.no_grad():
            latents = base(
                prompt=final_prompt, negative_prompt=neg,
                num_inference_steps=steps, denoising_end=0.8,
                guidance_scale=guid, width=w_val, height=h_val,
                generator=gen, output_type='latent',
            ).images
            image = refiner(
                prompt=final_prompt, negative_prompt=neg,
                num_inference_steps=steps, denoising_start=0.8,
                guidance_scale=guid, image=latents, generator=gen,
            ).images[0]
        elapsed = time.time() - t0

    except Exception:
        err = traceback.format_exc()
        with status_out:
            clear_output(wait=True)
            display(HTML(f'<pre style="color:#ef4444;font-size:11px;background:#1a0505;padding:12px;border-radius:8px;">{err}</pre>'))
        gen_btn.disabled    = False
        gen_btn.description = ' Generate Image'
        return

    fname = f'/content/generated_images/out_{int(time.time())}.png'
    image.save(fname)
    history.append((image, prompt, elapsed, used, res, style))
    if len(history) > MAX_HIST: history.pop(0)

    b64 = _b64(image)
    vram_used = torch.cuda.memory_allocated() / 1024**3

    with img_out:
        clear_output(wait=True)
        display(HTML(
            f'<div style="max-width:900px;">'
            f'<div style="background:#042918;border:1px solid #065f39;border-radius:10px;padding:12px 16px;margin-bottom:12px;">'
            f'<span style="color:#10b981;font-size:14px;font-weight:700;"> Done in {elapsed:.1f}s</span>'
            f'<span style="color:#64748b;font-size:12px;margin-left:16px;">{style} · {res} · {steps} steps · guidance {guid:.1f} · seed {used} · VRAM {vram_used:.1f} GB</span>'
            f'</div>'
            f'<img src="data:image/png;base64,{b64}" style="width:100%;border-radius:12px;border:1px solid #2a2a50;display:block;">'
            f'<a download="img_{used}.png" href="data:image/png;base64,{b64}"'
            f' style="display:block;margin-top:10px;padding:14px;text-align:center;'
            f'background:linear-gradient(135deg,#7c3aed,#4f46e5);color:#fff;'
            f'border-radius:10px;text-decoration:none;font-weight:700;font-size:15px;">⬇ Download PNG</a>'
            f'</div>'
        ))

    with status_out:
        clear_output(wait=True)
        display(HTML(f'<p style="color:#10b981;margin:0;">Saved to {fname}</p>'))

    render_history()
    gen_btn.disabled    = False
    gen_btn.description = ' Generate Image'

gen_btn.on_click(on_generate)

# ── Assemble layout ───────────────────────────────────────────────────────
display(w.VBox([
    title_html,
    w.HTML('<div class="sec-hdr">Prompt</div>'),
    prompt_box,
    w.HTML('<div class="sec-hdr">Style</div>'),
    style_row1, style_row2,
    w.HTML('<div class="sec-hdr">Negative Prompt</div>'),
    neg_box,
    w.HTML('<div class="sec-hdr">Settings</div>'),
    w.HBox([res_drop, steps_sl, guidance_sl, seed_box],
           layout=w.Layout(flex_wrap='wrap', gap='12px', align_items='center')),
    w.HTML('<div class="sec-hdr">Quick Prompts</div>'),
    ex_row,
    w.HTML('<div style="height:8px"></div>'),
    gen_btn,
    status_out,
    img_out,
    hist_out,
], layout=w.Layout(max_width='920px', padding='8px')))
print(" UI ready! Enter a prompt and click Generate Image.")
